# GeoFWI3D fold 0: confirm seeds 0, 1, 2
Five baseline-only jobs: UNETR LR6 seeds 1,2, then established 3D U-Net seeds 0,1,2. Same cap per run (182,178.12654316 GFLOPs), no halfway gate, no new BANDS runs. UNETR uses Dice+CE, XY D4, dropout .1 and LR 6e-4; 3D U-Net retains Dice-only, no augmentation, LR 3e-4. Final-budget Dice strictly >50% for every one of 12 model/seed cells is required. Reuse verified nnU-Net and SwinUNETR seeds 0–2 and archived UNETR seed0 (51.5067%). This completes only fold0, not the full three-fold study.


In [ ]:
from google.colab import drive, auth
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True, timeout_ms=180000)
from googleapiclient.discovery import build
import google.auth
credentials, _ = google.auth.default()
service = build('drive','v3',credentials=credentials,cache_discovery=False)
about = service.about().get(fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'filippofocaccia2@gmail.com', 'Mount/authenticate the requested account.'
quota=about['storageQuota']
if 'limit' in quota:
    assert int(quota['limit'])-int(quota['usage']) >= 8*1024**3, 'Keep at least 8 GiB free in Drive, including trash.'
from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,zipfile
HOME_DIR=Path('/content/drive/MyDrive/hippo_geofwi3d_baseline_seed_search')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_fold0_confirmation_lr6_v1'
PREVIOUS=HOME_DIR/'results_mixed_noaug_cap182178_v1'
BUNDLE=Path('/content/hippo_geofwi_fold0_confirmation_v1')
RESULTS.mkdir(parents=True,exist_ok=True)
BUNDLE.mkdir(exist_ok=True)
assert shutil.disk_usage('/content').free >= 24*1024**3, 'Need at least 24 GiB runtime disk space.'
print('Account:',about['user']['emailAddress'],'Results:',RESULTS)


## Verify and stage the frozen code and existing dataset
Reuse the data already stored on this account. Every upload and extracted file is hash checked.


In [ ]:
EXPECTED_UPLOADS={'geofwi3d_data_part01.zip': '32eaadf1a6abd61dd5bd922a8a085af4e6c876b34b915bbd09da93a3dbea2ca2', 'geofwi3d_data_part02.zip': '31adc6fa680b74a921f9457f24d2dfbb178aa1cef1d80baa152c5ee82a4e05bd', 'geofwi3d_data_part03.zip': 'e9c404631c47a918d1ca36c9166ea92141aa1b8264e332fbaa32b13299d8706a', 'geofwi3d_data_part04.zip': '1ef1e965ebefb571f441d986f2304edb4f2ab824b4bedb7dfbc6a494c4c1c279', 'geofwi3d_data_part05.zip': 'b99cbe88932b94bc382512c059661b48516beacd1dae116237ec411edc64fb58', 'geofwi3d_data_part06.zip': '6f0509326647688300459b05d5f0af5c10ecdde9c19fc6fc3ee05bc92fe8f183', 'geofwi3d_data_part07.zip': '3e1cf8a1cacd445c73bf1798ebd9c1b3c34f6f2cea0e2bf0ad378c6f12f09fb1', 'geofwi3d_data_part08.zip': 'b2b603f5ef55eb2822e64904ce7d178b7cd459c1258398de1f75cdedad5525ed', 'geofwi3d_data_part09.zip': '833fccd13996d6d6ddd118b6146334a902e4eceba2dedd26b798db64fcf6de38', 'geofwi3d_data_part10.zip': '99daf1d8f684ac47897aaa6933b69741e826935783bc42dbd02093efc299e235', 'geofwi3d_data_part11.zip': 'ce12340263414416fcd728e7fad43a2dbbbf4d26f22cc51273acb8df50680239', 'geofwi3d_data_part12.zip': 'ff0f922b1ae82afb2394725bcd0374b4eb273fe0333f5fbad4dbd52b540aadd2', 'geofwi3d_data_part13.zip': '9ede65d6da572c0186b51b838f46f7cd281bbd8b515aef813e502b59891cc880', 'geofwi3d_fold0_baselines_v1.zip': 'da00ad05c33235a6609e680ce3062ccabe7cd93fff233d3beea740fba91936f9'}
EXPECTED_MANIFEST='ed1142ad18b84ea311d74183d625ed76bc141cc45553714d1a88b5cc69b25253'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_UPLOADS.items():
    package=INPUT/name
    assert sha(package)==expected, 'Upload checksum differs: '+name
    local=Path('/content')/name
    if not local.exists() or sha(local)!=expected:shutil.copyfile(package,local)
    assert sha(local)==expected
    with zipfile.ZipFile(local) as z:
        assert len(z.namelist())==len(set(z.namelist()))
        for info in z.infolist():
            p=Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts
            assert (info.external_attr>>16)&0o170000 != 0o120000
            target=BUNDLE/p
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.extracting')
            with z.open(info) as src,tmp.open('wb') as dst:shutil.copyfileobj(src,dst)
            tmp.replace(target)
    print('Staged',name,flush=True)
assert sha(BUNDLE/'manifest.json')==EXPECTED_MANIFEST
for name,expected in json.loads((BUNDLE/'manifest.json').read_text()).items():assert sha(BUNDLE/name)==expected,name
for child in ['unetr','basic']:
    data=BUNDLE/child/'data'
    if not data.exists():data.symlink_to(BUNDLE/'data',target_is_directory=True)
    for name,expected in json.loads((BUNDLE/child/'data_manifest.json').read_text()).items():assert sha(BUNDLE/child/name)==expected,name
provenance=RESULTS/'provenance';provenance.mkdir(exist_ok=True)
shutil.copyfile(INPUT/'geofwi3d_fold0_baselines_v1.zip',provenance/'geofwi3d_fold0_baselines_v1.zip')
assert sha(provenance/'geofwi3d_fold0_baselines_v1.zip')==EXPECTED_UPLOADS['geofwi3d_fold0_baselines_v1.zip']
print('All data and code verified')


## Match the established A100-80GB environment
Each new model recipe passes a short GPU resume diagnostic before production. Diagnostic compute is separately recorded.


In [ ]:
VENV=Path('/content/hippo_geofwi_fold0_venv');PYTHON=VENV/'bin/python'
def command(args,log_name):
    with (RESULTS/log_name).open('a') as log:
        proc=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True)
        tail=[]
        def drain():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                tail.append(line);del tail[:-40]
        try:drain();code=proc.wait()
        except KeyboardInterrupt:
            # Signal the orchestrator only; it stops the active trainer at a safe boundary.
            proc.send_signal(signal.SIGINT)
            print('Saving recovery to Drive. Please wait; do not interrupt again.')
            drain();code=proc.wait()
        finally:proc.stdout.close()
        if code:raise RuntimeError('Process failed; recovery retained.\n'+''.join(tail))
if not PYTHON.exists():
    command([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup.log')
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    command([PYTHON,'-m','ensurepip','--upgrade'],'setup.log')
command([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'unetr/requirements-colab.txt'],'setup.log')
subprocess.run(['nvidia-smi'],check=True)


## Validate all seven reused baseline references
Recompute Dice from confusion counts, check all archive hashes, folds/seeds, recipes and full-budget completion. No new training in this cell.


In [ ]:
QUEUE=BUNDLE/'queue.py'
BASE_COMMAND=[PYTHON,'-u',QUEUE,'--bundle',BUNDLE,'--durable',RESULTS,'--previous-results',PREVIOUS,'--hours','6']
command(BASE_COMMAND+['--verify-only'],'verify_references.log')


## Run the five authorized baselines sequentially
UNETR seeds1,2 then 3D U-Net seeds0,1,2. Re-running resumes or skips verified completed results. Every finished run is archived and verified before bulky checkpoints are retired. An under-50 score is retained and cannot qualify fold0; it does not silently trigger a new seed search.


In [ ]:
command(BASE_COMMAND,'fold0_run.log')


## Fold-0 status
`FOLD0_QUALIFIED.json` is created only when all 12 baseline cells exceed 50%. `FIVE_RUNS_FINISHED.json` records completion even if qualification fails.


In [ ]:
print((RESULTS/'fold0_status.json').read_text())
